# 05 · Retrieval Metrics — expanded, with worked examples
Beyond Precision@K and Recall@K: **MRR**, **MAP**, **nDCG**, and **F1** — each with a hand-worked example so you can verify every number.

## The running example
```
  relevant docs = {A, B, C}   (3 exist)
  ranked results: 1:A(rel) 2:X 3:B(rel) 4:Y 5:C(rel)
```

In [ ]:
ranked   = ["A","X","B","Y","C"]
relevant = {"A","B","C"}

def p_at_k(r, rel, k): return sum(d in rel for d in r[:k]) / k
def r_at_k(r, rel, k): return sum(d in rel for d in r[:k]) / len(rel)

print("Precision@K / Recall@K:")
for k in [1,3,5]:
    print(f"  k={k}:  P@{k}={p_at_k(ranked,relevant,k):.3f}  R@{k}={r_at_k(ranked,relevant,k):.3f}")

## MRR — how high was the FIRST hit
```
  first relevant is at rank 1  ->  RR = 1/1 = 1.0
  MRR = mean of RR over many queries
```

In [ ]:
def rr(r, rel):
    for i,d in enumerate(r,1):
        if d in rel: return 1.0/i
    return 0.0
print("RR =", rr(ranked, relevant))

## MAP — Mean Average Precision (rewards putting ALL relevant high)
```
  AP = average of Precision@k taken ONLY at the ranks where a relevant doc appears
  rank1 A rel -> P@1 = 1/1 = 1.00
  rank3 B rel -> P@3 = 2/3 = 0.667
  rank5 C rel -> P@5 = 3/5 = 0.600
  AP = (1.00 + 0.667 + 0.600) / 3 = 0.756
  MAP = mean AP over all queries
```

In [ ]:
def average_precision(r, rel):
    hits=0; s=0.0
    for i,d in enumerate(r,1):
        if d in rel:
            hits+=1; s += hits/i
    return s/len(rel) if rel else 0.0
print("AP =", round(average_precision(ranked, relevant),3), "(matches 0.756 above)")

## nDCG — graded relevance + position discount
```
  DCG  = sum over ranks of  rel_i / log2(i+1)     (lower ranks discounted)
  IDCG = DCG of the IDEAL ordering (all relevant first)
  nDCG = DCG / IDCG    (1.0 = perfect ordering)
```
Unlike P/R, nDCG supports **graded** relevance (a doc can be 'very' vs 'somewhat' relevant), and it cares about *order*, not just presence in the top-k.

In [ ]:
import numpy as np
def dcg(rels):                      # rels = relevance grades in rank order
    return sum(rel/np.log2(i+2) for i,rel in enumerate(rels))
# binary relevance for our example: A,X,B,Y,C -> 1,0,1,0,1
gains=[1,0,1,0,1]
ideal=sorted(gains, reverse=True)   # 1,1,1,0,0
ndcg = dcg(gains)/dcg(ideal)
print("DCG =", round(dcg(gains),3), " IDCG =", round(dcg(ideal),3), " nDCG =", round(ndcg,3))

## F1 — one number balancing precision & recall
```
  F1@k = 2 * P@k * R@k / (P@k + R@k)
```

In [ ]:
def f1_at_k(r, rel, k):
    p,rc = p_at_k(r,rel,k), r_at_k(r,rel,k)
    return 2*p*rc/(p+rc) if (p+rc) else 0.0
for k in [1,3,5]:
    print(f"  F1@{k} = {f1_at_k(ranked,relevant,k):.3f}")

## Which metric answers which question
| Metric | Answers | Use when |
|---|---|---|
| P@K | of the top-K, how many are relevant? | you show K results |
| R@K | of all relevant, how many are in top-K? | coverage matters |
| MRR | how high is the FIRST hit? | one right answer |
| MAP | are ALL relevant ranked high? | several right answers, order matters |
| nDCG | graded relevance, order-aware | some docs 'more' relevant than others |
| F1 | balance P and R in one number | you want a single score |

**Observe:** all six are computed from the same ranked list + relevance labels — different lenses on the same data. Pick the lens that matches what your users actually need.